In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# 方針
これまで作ってきた特徴量を追加し、Pipelineの中に組み込んでいく。そして、いくつかのモデルのスコアをパラメータの最適解を探りながら算出し、それらモデルのスコア比較を行い、最適モデルを見つける

* 「03_eda」,「08_feature_engineering」ファイルより、新特徴量として、”Sex_Pclass"、"logFare"、"len_Fam"、"Title"、”surb_rank"、"has_cabin"、"cabin_deck"の7つを加える
* 「07_pipeline」の中に組み込んでいく

In [2]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# データの読み込み

In [3]:
train_csv = pd.read_csv("/kaggle/input/competitions/titanic/train.csv").set_index("PassengerId")
test_csv = pd.read_csv("/kaggle/input/competitions/titanic/test.csv").set_index("PassengerId")

y = train_csv.Survived
X = train_csv.drop("Survived", axis=1)

# カスタム特徴量を作るトランスフォーマー
「08_feature_engineering」ファイルでつくりだした特徴量である、"Title","surv_rank","has_cabin","cabin_deck"の4つを追加していく

※「len_fam」について
>前回まではlen_famの値として,家族の合計によって"alone"、"basic"、"large"の3つに分けていたが、これらの値は本来であれば大小関係を数値によって学習できるものになるので、カテゴリとして処理するよりも数値列として学習させたほうが良いと判断。よって、今回は｛"alone":1,"basic":2,"large":3｝として、それぞれ重みを付けた形で再定義した。

In [4]:
class TitanicFeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass #paramatoeはない

    def fit(self,X,y=None):
        return self #今回は特徴量作成に統計データ使わないのでfit要らない

    def transform(self,X): #ここから新特徴量の作成
        X_new = X.copy()

        #"Sex_Pclass"の作成
        X_new["Sex_Pclass"] = (
            X_new["Sex"].astype(str) + "_" +
            X_new["Pclass"].astype(str)
        )

        #"logFare"の作成
        X_new["logFare"] = np.log1p(X_new["Fare"])

        #"len_fam"の作成
        family = X_new["Parch"] + X_new["SibSp"]
        X_new["len_fam"] = family.apply(
            lambda x: (
                0 if x == 0 else
                1 if 1<=x<=3 else
                2 
            )
        )

        #"Title"の作成
        X_new["Title"] = X_new["Name"].str.extract(" ([A-Za-z]+)\.",expand = False)
        X_new["Title"] = X_new["Title"].replace(["Mlle","Ms"],"Miss")
        X_new["Title"] = X_new["Title"].replace("Mme","Mrs")
        rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
        X_new["Title"] = X_new["Title"].replace(rare_titles,"Rare")

        #”Age"の再定義。年齢の欠損を敬称別の中央値の年齢で埋める
        X_new["Age"] = X_new["Age"].fillna(X_new.groupby("Title")["Age"].transform("median"))

        #"surv_rank"を作成、デフォルトで0点上記の条件以外の部分
        condition_3points = (
            ((X_new["Sex"]=="male") & (X_new["Age"]<12)) |
            ((X_new["Sex"]=="female") & ((X_new["Age"]<6) |
                                    (X_new["Age"]>=48)))
        )
        condition_2points = (
            (X_new["Sex"]=="female") & (X_new["Age"]>=12) & (X_new["Age"]<48)
        )
        condition_1point = (
            (X_new["Sex"]=="female") & (X_new["Age"]>=6) & (X_new["Age"]<12)
        )
        conditions = [condition_3points,condition_2points,condition_1point]
        points = [3,2,1]
        X_new["surv_rank"] = np.select(conditions,points,default=0)

        #"has_cabin"を作成
        X_new["has_cabin"] = X_new["Cabin"].notnull().astype(int)

        #"cabin_deck"を作成
        X_new["cabin_deck"] = X_new["Cabin"].str[0].fillna("Unknown")
        X_new.loc[X_new["cabin_deck"].isin(["G","T"]),"cabin_deck"] = "Unknown"
        
        return X_new
        

<>:31: SyntaxWarning: invalid escape sequence '\.'
<>:31: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_16/3144778359.py:31: SyntaxWarning: invalid escape sequence '\.'
  X_new["Title"] = X_new["Name"].str.extract(" ([A-Za-z]+)\.",expand = False)


# Pipelineの構築

In [5]:
#数値列に対する前処理
numerical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

#カテゴリ列に対する前処理
categorical_transformer = Pipeline(
    steps = [
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output = False))
    ]
)

#使用する列を明示、"len_fam"は今回はnum_colsに移動している
num_cols = ["Age","SibSp","Parch","Pclass","logFare","len_fam","surv_rank"]
cat_cols = ["Sex","Embarked","Sex_Pclass","has_cabin","cabin_deck","Title"]

#前処理の結合
preprocessor = ColumnTransformer(
    transformers=[
        ("num",numerical_transformer,num_cols),
        ("cat",categorical_transformer,cat_cols)
    ]
)

#すべての工程をPipelineにまとめる
my_pipeline = Pipeline(
    steps = [
        ("feature_engineering",TitanicFeatureEngineering()),
        ("preprocessor",preprocessor),
        ("model",XGBClassifier(
            n_estimators = 1000,
            learning_rate = 0.01,
            subsample = 0.8,
            colsample_bytree = 0.8,
            max_depth = 3,
            gamma=1,
            min_child_weight = 2,
            random_state = 10,
        ))
    ]
)

In [6]:
my_pipeline.fit(X,y)
X_train,X_valid,y_train,y_valid = train_test_split(X,y,test_size=0.2,random_state=0)

val_accuracy = my_pipeline.score(X_valid,y_valid)
print(val_accuracy)

preds = my_pipeline.predict(test_csv)

output = pd.DataFrame({"PassengerId":test_csv.index,"Survived":preds})
output.to_csv("submission.csv",index=False)

0.9106145251396648
